
Python Logging Lab: Building an Application Performance Monitor
================================================================
Learn how to use Python's logging library by creating a simple performance
monitoring system that tracks function execution times and system metrics.


In [15]:
import logging
import time
import random
import json
from datetime import datetime
from functools import wraps

### PART 1: Setting Up Advanced Logging Configuration

In [16]:
# Configure logging with multiple formatters for different purposes
def setup_logging():
    """Configure a comprehensive logging system with multiple handlers"""
    
    # Create formatters for different log types
    detailed_formatter = logging.Formatter(
        '%(asctime)s | %(name)-15s | %(levelname)-8s | %(funcName)-20s | %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    )
    
    simple_formatter = logging.Formatter(
        '%(levelname)s: %(message)s'
    )
    
    json_formatter = logging.Formatter(
        '{"time": "%(asctime)s", "level": "%(levelname)s", "module": "%(name)s", "message": "%(message)s"}'
    )
    
    # Create handlers
    console_handler = logging.StreamHandler()
    console_handler.setLevel(logging.INFO)
    console_handler.setFormatter(simple_formatter)
    
    file_handler = logging.FileHandler('performance_monitor.log')
    file_handler.setLevel(logging.DEBUG)
    file_handler.setFormatter(detailed_formatter)
    
    error_handler = logging.FileHandler('errors.log')
    error_handler.setLevel(logging.ERROR)
    error_handler.setFormatter(detailed_formatter)
    
    # Configure root logger
    root_logger = logging.getLogger()
    root_logger.setLevel(logging.DEBUG)
    root_logger.addHandler(console_handler)
    root_logger.addHandler(file_handler)
    root_logger.addHandler(error_handler)
    
    return root_logger

### PART 2: Creating Specialized Loggers for Different Components

In [17]:
# Create module-specific loggers
performance_logger = logging.getLogger('performance')
database_logger = logging.getLogger('database')
api_logger = logging.getLogger('api')
security_logger = logging.getLogger('security')

### PART 3: Performance Monitoring Decorator with Logging

In [18]:
def log_performance(threshold_ms=100):
    """Decorator to log function execution time"""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            start_time = time.time()
            logger = performance_logger
            
            logger.debug(f"Starting execution of {func.__name__}")
            
            try:
                result = func(*args, **kwargs)
                execution_time_ms = (time.time() - start_time) * 1000
                
                # Log based on execution time
                if execution_time_ms > threshold_ms:
                    logger.warning(
                        f"Slow function: {func.__name__} took {execution_time_ms:.2f}ms "
                        f"(threshold: {threshold_ms}ms)"
                    )
                else:
                    logger.info(f"{func.__name__} completed in {execution_time_ms:.2f}ms")
                
                return result
                
            except Exception as e:
                execution_time_ms = (time.time() - start_time) * 1000
                logger.error(
                    f"Function {func.__name__} failed after {execution_time_ms:.2f}ms: {str(e)}",
                    exc_info=True
                )
                raise
                
        return wrapper
    return decorator

### PART 4: Simulated Application Components with Logging

In [19]:
@log_performance(threshold_ms=50)
def simulate_database_query(query_type="SELECT"):
    """Simulate a database operation with logging"""
    database_logger.debug(f"Executing {query_type} query")
    
    # Simulate random query time
    query_time = random.uniform(0.01, 0.15)
    time.sleep(query_time)
    
    # Simulate occasional errors
    if random.random() < 0.1:  # 10% chance of error
        database_logger.error(f"Database connection timeout for {query_type} query")
        raise ConnectionError("Database connection failed")
    
    rows_affected = random.randint(1, 100)
    database_logger.info(f"{query_type} query affected {rows_affected} rows")
    return rows_affected

@log_performance(threshold_ms=100)
def simulate_api_call(endpoint="/users", method="GET"):
    """Simulate an API call with detailed logging"""
    request_id = f"REQ-{random.randint(1000, 9999)}"
    api_logger.info(f"[{request_id}] {method} {endpoint}")
    
    # Simulate API response time
    response_time = random.uniform(0.05, 0.2)
    time.sleep(response_time)
    
    # Simulate different response codes
    status_codes = [200, 200, 200, 201, 400, 404, 500]  # Weighted towards success
    status_code = random.choice(status_codes)
    
    if status_code >= 500:
        api_logger.error(f"[{request_id}] Server error: {status_code}")
        raise Exception(f"API returned {status_code}")
    elif status_code >= 400:
        api_logger.warning(f"[{request_id}] Client error: {status_code}")
    else:
        api_logger.info(f"[{request_id}] Success: {status_code}")
    
    return {"status": status_code, "request_id": request_id}

def simulate_user_authentication(username, password):
    """Simulate user authentication with security logging"""
    security_logger.info(f"Authentication attempt for user: {username}")
    
    # Simulate authentication check
    time.sleep(0.05)
    
    # Simulate authentication results
    if username == "admin" and password == "correct_password":
        security_logger.info(f"Successful authentication for user: {username}")
        return True
    else:
        security_logger.warning(
            f"Failed authentication attempt for user: {username} from IP: 192.168.1.{random.randint(1, 255)}"
        )
        return False

### PART 5: Custom Log Filter for Sensitive Data

In [20]:
class SensitiveDataFilter(logging.Filter):
    """Filter to mask sensitive information in logs"""
    
    SENSITIVE_PATTERNS = ['password', 'token', 'api_key', 'secret']
    
    def filter(self, record):
        # Mask sensitive data in log messages
        message = record.getMessage()
        for pattern in self.SENSITIVE_PATTERNS:
            if pattern in message.lower():
                # Replace actual values with masked version
                record.msg = record.msg.replace(pattern, f"{pattern}=***MASKED***")
        return True

### PART 6: Log Analysis and Reporting

In [21]:
class LogAnalyzer:
    """Analyze log patterns and generate reports"""
    
    def __init__(self, log_file='performance_monitor.log'):
        self.log_file = log_file
        self.logger = logging.getLogger('analyzer')
    
    def analyze_performance_metrics(self):
        """Analyze performance metrics from logs"""
        metrics = {
            'slow_functions': [],
            'errors': [],
            'warnings': []
        }
        
        try:
            with open(self.log_file, 'r') as f:
                for line in f:
                    if 'Slow function' in line:
                        metrics['slow_functions'].append(line.strip())
                    elif 'ERROR' in line:
                        metrics['errors'].append(line.strip())
                    elif 'WARNING' in line:
                        metrics['warnings'].append(line.strip())
            
            self.logger.info(f"Analysis complete: {len(metrics['errors'])} errors found")
            return metrics
            
        except FileNotFoundError:
            self.logger.warning(f"Log file {self.log_file} not found")
            return metrics

### PART 7: Context Manager for Timed Operations

In [22]:
class TimedOperation:
    """Context manager for timing and logging operations"""
    
    def __init__(self, operation_name, logger=None):
        self.operation_name = operation_name
        self.logger = logger or logging.getLogger(__name__)
        self.start_time = None
    
    def __enter__(self):
        self.start_time = time.time()
        self.logger.info(f"Starting operation: {self.operation_name}")
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        duration = time.time() - self.start_time
        
        if exc_type:
            self.logger.error(
                f"Operation {self.operation_name} failed after {duration:.3f}s: {exc_val}"
            )
        else:
            self.logger.info(
                f"Operation {self.operation_name} completed in {duration:.3f}s"
            )
        
        return False  # Don't suppress exceptions

### PART 8: Main Application Demo

In [23]:
def run_monitoring_demo():
    """Run the complete monitoring demonstration"""
    
    # Setup logging
    setup_logging()
    
    # Add sensitive data filter to security logger
    sensitive_filter = SensitiveDataFilter()
    security_logger.addFilter(sensitive_filter)
    
    main_logger = logging.getLogger('main')
    main_logger.info("=" * 60)
    main_logger.info("Application Performance Monitor Started")
    main_logger.info("=" * 60)
    
    # Demonstrate different logging scenarios
    
    # 1. Database Operations
    print("\n1. Testing Database Operations...")
    for i in range(3):
        try:
            rows = simulate_database_query(random.choice(["SELECT", "UPDATE", "INSERT"]))
            main_logger.debug(f"Database operation {i+1} completed")
        except ConnectionError as e:
            main_logger.error(f"Database operation {i+1} failed: {e}")
    
    # 2. API Calls
    print("\n2. Testing API Calls...")
    endpoints = ["/users", "/products", "/orders", "/analytics"]
    for endpoint in endpoints:
        try:
            response = simulate_api_call(endpoint, random.choice(["GET", "POST", "PUT"]))
            main_logger.debug(f"API call to {endpoint} completed")
        except Exception as e:
            main_logger.error(f"API call to {endpoint} failed: {e}")
    
    # 3. Authentication Attempts
    print("\n3. Testing Authentication...")
    test_credentials = [
        ("admin", "correct_password"),
        ("user1", "wrong_password"),
        ("hacker", "attempt123")
    ]
    
    for username, password in test_credentials:
        # This will demonstrate the sensitive data filter
        security_logger.debug(f"Checking credentials - username: {username}, password: {password}")
        result = simulate_user_authentication(username, password)
    
    # 4. Timed Operations
    print("\n4. Testing Timed Operations...")
    with TimedOperation("Data Processing", main_logger):
        # Simulate data processing
        time.sleep(random.uniform(0.5, 1.5))
        main_logger.info("Processing 1000 records...")
    
    # 5. Simulate an error scenario
    print("\n5. Testing Error Handling...")
    try:
        with TimedOperation("Risky Operation", main_logger):
            if random.random() > 0.5:
                raise ValueError("Simulated error in risky operation")
    except ValueError:
        main_logger.error("Risky operation failed as expected", exc_info=True)
    
    # 6. Log Analysis
    print("\n6. Analyzing Logs...")
    analyzer = LogAnalyzer()
    metrics = analyzer.analyze_performance_metrics()
    
    # Summary Report
    print("\n" + "=" * 60)
    print("MONITORING SUMMARY")
    print("=" * 60)
    main_logger.info(f"Total Errors: {len(metrics['errors'])}")
    main_logger.info(f"Total Warnings: {len(metrics['warnings'])}")
    main_logger.info(f"Slow Functions: {len(metrics['slow_functions'])}")
    
    main_logger.info("=" * 60)
    main_logger.info("Application Performance Monitor Completed")
    main_logger.info("=" * 60)

### PART 9: Custom Log Levels

In [24]:
# Define custom log levels for specific monitoring needs
PERFORMANCE_CRITICAL = 35  # Between WARNING (30) and ERROR (40)
logging.addLevelName(PERFORMANCE_CRITICAL, "PERF_CRITICAL")

def perf_critical(self, message, *args, **kwargs):
    """Add custom performance critical logging method"""
    if self.isEnabledFor(PERFORMANCE_CRITICAL):
        self._log(PERFORMANCE_CRITICAL, message, args, **kwargs)

# Add the custom method to Logger class
logging.Logger.perf_critical = perf_critical


In [25]:
if __name__ == "__main__":
    run_monitoring_demo()
    
    print("\n" + "=" * 60)
    print("Check the following files for detailed logs:")
    print("  - performance_monitor.log (all logs)")
    print("  - errors.log (errors only)")
    print("=" * 60)

INFO: ============================================================
INFO: ============================================================
INFO: Application Performance Monitor Started
INFO: Application Performance Monitor Started
INFO: ============================================================
INFO: ============================================================
INFO: INSERT query affected 4 rows
INFO: INSERT query affected 4 rows



1. Testing Database Operations...


INFO: SELECT query affected 56 rows
INFO: SELECT query affected 56 rows
ERROR: Database connection timeout for INSERT query
ERROR: Database connection timeout for INSERT query
ERROR: Function simulate_database_query failed after 131.78ms: Database connection failed
Traceback (most recent call last):
  File "/var/folders/rg/0rpm6k950ml5nkp48wh2558w0000gn/T/ipykernel_45328/1607424925.py", line 12, in wrapper
    result = func(*args, **kwargs)
  File "/var/folders/rg/0rpm6k950ml5nkp48wh2558w0000gn/T/ipykernel_45328/3023672175.py", line 13, in simulate_database_query
    raise ConnectionError("Database connection failed")
ConnectionError: Database connection failed
ERROR: Function simulate_database_query failed after 131.78ms: Database connection failed
Traceback (most recent call last):
  File "/var/folders/rg/0rpm6k950ml5nkp48wh2558w0000gn/T/ipykernel_45328/1607424925.py", line 12, in wrapper
    result = func(*args, **kwargs)
  File "/var/folders/rg/0rpm6k950ml5nkp48wh2558w0000gn/T/ipyk


2. Testing API Calls...


INFO: [REQ-1259] Success: 200
INFO: [REQ-1259] Success: 200
INFO: [REQ-4464] POST /orders
INFO: [REQ-4464] POST /orders
INFO: [REQ-4464] Success: 200
INFO: [REQ-4464] Success: 200
INFO: [REQ-5507] GET /analytics
INFO: [REQ-5507] GET /analytics
INFO: [REQ-5507] Success: 200
INFO: [REQ-5507] Success: 200
INFO: Authentication attempt for user: admin
INFO: Authentication attempt for user: admin
INFO: Successful authentication for user: admin
INFO: Successful authentication for user: admin
INFO: Authentication attempt for user: user1
INFO: Authentication attempt for user: user1
INFO: Authentication attempt for user: hacker
INFO: Authentication attempt for user: hacker
INFO: Starting operation: Data Processing
INFO: Starting operation: Data Processing



3. Testing Authentication...

4. Testing Timed Operations...


INFO: Processing 1000 records...
INFO: Processing 1000 records...
INFO: Operation Data Processing completed in 1.403s
INFO: Operation Data Processing completed in 1.403s
INFO: Starting operation: Risky Operation
INFO: Starting operation: Risky Operation
ERROR: Operation Risky Operation failed after 0.002s: Simulated error in risky operation
ERROR: Operation Risky Operation failed after 0.002s: Simulated error in risky operation
ERROR: Risky operation failed as expected
Traceback (most recent call last):
  File "/var/folders/rg/0rpm6k950ml5nkp48wh2558w0000gn/T/ipykernel_45328/4009912079.py", line 62, in run_monitoring_demo
    raise ValueError("Simulated error in risky operation")
ValueError: Simulated error in risky operation
ERROR: Risky operation failed as expected
Traceback (most recent call last):
  File "/var/folders/rg/0rpm6k950ml5nkp48wh2558w0000gn/T/ipykernel_45328/4009912079.py", line 62, in run_monitoring_demo
    raise ValueError("Simulated error in risky operation")
ValueEr


5. Testing Error Handling...

6. Analyzing Logs...

MONITORING SUMMARY

Check the following files for detailed logs:
  - performance_monitor.log (all logs)
  - errors.log (errors only)
